In [6]:
#Step 1: Data Generation & Setup / 步骤一：环境设置与数据生成

#What it does (做什么):
#Initialize the environment, set basic parameters, and simulate distributed data across K sites. The data contains a few true signal features and many noise features, with slight correlation among features.
#初始化环境，设置基本参数，并模拟生成分布在 K 个站点的数据。数据中包含少数真实的信号特征和大量噪音特征，且特征之间存在轻微相关性。

In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Experimental basic parameters / 实验基本参数
np.random.seed(42)
K = 3         # Number of sites / 站点数量
n = 300       # Sample size per site / 每个站点的样本量
p = 200       # Total feature dimension (high-dimensional) / 特征总维度 (高维)
p1 = 8        # Number of truly significant features / 真正有信号的特征数

# True coefficients: the first 8 features are non-zero, randomly assigned ±0.8 signals, others are all 0 / 真实系数：前 8 个特征非零，随机赋予 +0.8 或 -0.8 的信号，其余全为 0
beta_true = np.zeros(p)
beta_true[:p1] = np.random.choice([-1, 1], p1) * 0.8

sites_data = []
for k in range(K):
    X = np.random.randn(n, p)
    # Simulate slight collinearity of real-world data / 模拟真实世界数据的轻微共线性
    for i in range(p):
        for j in range(i+1, p):
            if np.abs(i-j) < 5:
                X[:,i] += 0.2 * X[:,j]
    
    # Generate binary classification labels y based on logistic regression model / 根据逻辑回归模型生成二分类标签 y
    lin_pred = X @ beta_true
    prob = 1 / (1 + np.exp(-lin_pred))
    y = (np.random.rand(n) < prob).astype(int)
    sites_data.append((X, y))

print(f"✅ Successfully generated {K} sites of data. Each site dimension: X={sites_data[0][0].shape}, y={sites_data[0][1].shape} / 成功生成 {K} 个站点的数据。每个站点维度: X={sites_data[0][0].shape}, y={sites_data[0][1].shape}")

✅ Successfully generated 3 sites of data. Each site dimension: X=(300, 200), y=(300,) / 成功生成 3 个站点的数据。每个站点维度: X=(300, 200), y=(300,)


In [8]:
#Step 2: Stage I - Local DP-SIS / 步骤二：本地差分隐私边际粗筛

#What it does (做什么):
#Each site calculates the marginal correlation for all features, applies a threshold clip to bound the sensitivity, adds Gaussian noise for privacy (GDP), and selects the top d features as its local candidate set.
#每个站点计算所有特征的边际相关性，通过截断来界定敏感度，加入高斯差分隐私噪声（GDP），然后选出得分最高的前 d 个特征作为本地候选集。

In [9]:
# DP and algorithm hyperparameters / DP及算法超参数
mu_total = 1.0                # Total GDP privacy budget / 总 GDP 隐私预算
mu1 = mu_total / np.sqrt(2)   # Budget allocated for Stage 1 / 阶段一分配的预算
mu2 = mu_total / np.sqrt(2)   # Budget allocated for Stage 2 / 阶段二分配的预算
d = 25                        # Number of locally selected features in Stage 1 / 阶段一本地截取的特征数
B = 2.0                       # Extreme value truncation boundary / 极值截断边界

local_support_sets = []

for k, (X, y) in enumerate(sites_data):
    # 1. Extreme value truncation to limit sensitivity / 1. 极值截断限制敏感度
    X_clip = np.clip(X, -B, B)
    n_samples = X_clip.shape[0]
    
    # 2. Calculate marginal scores (continuous real numbers) / 2. 计算边际得分 (连续的实数)
    omega = np.abs(X_clip.T @ y) / n_samples
    
    # 3. Calculate sensitivity and inject GDP noise / 3. 计算敏感度并注入 GDP 噪声
    Delta1 = 2 * B**2 / n_samples
    sigma1 = Delta1 / mu1
    noise = np.random.normal(0, sigma1, size=p)
    omega_tilde = omega + noise
    
    # 4. Sort by the noisy scores and select the top d features / 4. 根据加噪后的得分排序，截取前 d 个
    top_idx = np.argsort(omega_tilde)[-d:]
    local_support_sets.append(set(top_idx))
    
    n_true = len(set(top_idx) & set(range(p1)))
    print(f"Site {k+1} Stage 1 Screening: Selected {len(top_idx)} features, including true features {n_true}/{p1} / 站点 {k+1} 阶段一筛选: 选出 {len(top_idx)} 个特征，包含真实特征 {n_true}/{p1}")

Site 1 Stage 1 Screening: Selected 25 features, including true features 5/8 / 站点 1 阶段一筛选: 选出 25 个特征，包含真实特征 5/8
Site 2 Stage 1 Screening: Selected 25 features, including true features 6/8 / 站点 2 阶段一筛选: 选出 25 个特征，包含真实特征 6/8
Site 3 Stage 1 Screening: Selected 25 features, including true features 6/8 / 站点 3 阶段一筛选: 选出 25 个特征，包含真实特征 6/8


In [10]:
# Step 3: Majority Voting Mechanism / 步骤三：多数表决抗噪机制
# What it does (做什么):
# Instead of taking a naive union of the support sets, each site counts how many times a feature appears in the sets of other sites. It keeps only the features that receive at least v votes (here v=2).
# 各站点不再简单地取并集，而是统计某个特征在其他站点的候选集中出现的次数。只有得票数大于等于 v（这里 v=2）的特征才会被保留，形成鲁棒的支持集。

In [11]:
robust_support_sets = []

for k in range(K):
    # Collect support sets of all sites except itself / 收集除了自身以外其他所有站点的支持集
    other_supports = []
    for i in range(K):
        if i != k:
            other_supports.extend(list(local_support_sets[i]))
    
    # Count the occurrence frequency of each feature / 统计特征出现的频次
    freq = {}
    for feat in other_supports:
        freq[feat] = freq.get(feat, 0) + 1
        
    # Majority voting mechanism: when K=3, there are 2 other sites, only features appearing >= 2 times are retained
    # 多数表决机制：K=3 时，其余共2个站点，特征需出现次数≥2（全票通过）才予以保留
    S_minus_k = [feat for feat, count in freq.items() if count >= 2]
    robust_support_sets.append(S_minus_k)
    
    n_true_union = len(set(S_minus_k) & set(range(p1)))
    print(f"Site {k+1} Stage 2 Voting Union S_(-k): Dimension reduced to = {len(S_minus_k)}, including true features {n_true_union} / 站点 {k+1} 阶段二表决并集 S_(-k): 维度骤降至 = {len(S_minus_k)}，包含真实特征 {n_true_union}")

Site 1 Stage 2 Voting Union S_(-k): Dimension reduced to = 9, including true features 6 / 站点 1 阶段二表决并集 S_(-k): 维度骤降至 = 9，包含真实特征 6
Site 2 Stage 2 Voting Union S_(-k): Dimension reduced to = 7, including true features 5 / 站点 2 阶段二表决并集 S_(-k): 维度骤降至 = 7，包含真实特征 5
Site 3 Stage 2 Voting Union S_(-k): Dimension reduced to = 8, including true features 5 / 站点 3 阶段二表决并集 S_(-k): 维度骤降至 = 8，包含真实特征 5


In [12]:
# Step 4: Define the Regularized GLM Helper / 步骤四：定义正则化去偏辅助函数
# What it does (做什么):
# Defines a Logistic Regression function with L2 penalty (Ridge). It returns the fitted coefficients, the Hessian matrix, and crucially, the unpenalized gradient for the final debiasing step.
# 定义了一个带有 L2 惩罚项（Ridge）的逻辑回归函数。它返回拟合系数、海森矩阵（Hessian），以及用于最后去偏步骤的无惩罚梯度。

In [13]:
def fit_glm_logistic_regularized(X, y, lambda_reg=0.1):
    X_with_const = np.hstack([np.ones((X.shape[0], 1)), X]) # Add constant intercept term / 增加常数截距项
    
    # Negative log-likelihood loss function + L2 regularization / 负对数似然损失函数 + L2正则化
    def neg_log_likelihood(beta):
        lin = X_with_const @ beta
        mu = 1 / (1 + np.exp(-lin))
        eps = 1e-8
        loss = -np.mean(y * np.log(mu+eps) + (1-y)*np.log(1-mu+eps))
        reg = 0.5 * lambda_reg * np.sum(beta[1:]**2) # Do not penalize the intercept / 不惩罚截距
        return loss + reg
        
    def grad(beta):
        lin = X_with_const @ beta
        mu = 1 / (1 + np.exp(-lin))
        g = - (X_with_const.T @ (y - mu)) / len(y)
        g[1:] += lambda_reg * beta[1:] 
        return g

    beta0 = np.zeros(X_with_const.shape[1])
    res = minimize(neg_log_likelihood, beta0, method='BFGS', jac=grad, options={'maxiter': 500, 'disp': False})
    beta_hat = res.x
    
    # 1. Calculate Hessian matrix H / 1. 计算海森矩阵 H
    lin = X_with_const @ beta_hat
    mu = 1 / (1 + np.exp(-lin))
    W = mu * (1 - mu)
    H = (X_with_const.T * W) @ X_with_const / len(y)
    
    # 2. Calculate the unpenalized raw gradient (core of debiasing) / 2. 计算无惩罚的原始梯度 (这是去偏的核心)
    unpenalized_grad = - (X_with_const.T @ (y - mu)) / len(y)
    
    return beta_hat, H, unpenalized_grad

print("✅ Regularized GLM helper function loaded successfully. / 正则化 GLM 辅助函数加载完成。")

✅ Regularized GLM helper function loaded successfully. / 正则化 GLM 辅助函数加载完成。


In [14]:
# Why (为什么这么做):
# Without L2 regularization, standard MLE gradients converge exactly to 0, rendering the debiasing step mathematically useless. By fitting a Ridge model, the unpenalized gradient is non-zero, allowing the debiased Lasso formula (β^​init​−H−1×grad) to actually correct the shrinkage bias and restore asymptotic normality.
# 如果没有 L2 正则化，标准 MLE 收敛时的梯度永远是 0，这会导致去偏公式彻底失效。加入 Ridge 惩罚后，无惩罚梯度将不为 0，这才能让去偏公式（β^_(init)​−H^(−1)×grad）真正发挥作用，修正收缩偏差并恢复渐近正态性。

In [15]:
# Step 5: Privatized Debiasing & Upload / 步骤五：私有化去偏与二次加噪
# What it does (做什么):
# Each site runs the regularized GLM only on the robust support set. It applies the debiasing formula, and adds a global, uniform GDP noise to all features to protect the coefficient values.
# 每个站点仅在鲁棒支持集上运行正则化 GLM。应用去偏公式后，向所有特征统一加入全局 GDP 噪声，以保护系数本身的隐私。

In [17]:
coeffs_perturbed = []

for k, (X, y) in enumerate(sites_data):
    S_minus_k = robust_support_sets[k]
    
    if len(S_minus_k) == 0:
        coeffs_perturbed.append({})
        continue
        
    X_low = X[:, S_minus_k]
    
    # 1. Obtain biased estimates and necessary matrices / 1. 获取带有偏估计和必要矩阵
    beta_init, H, unpenalized_grad = fit_glm_logistic_regularized(X_low, y, lambda_reg=0.2)
    
    try:
        H_inv = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        H_inv = np.linalg.pinv(H)
    
    # 2. Perform the actual debiasing operation / 2. 执行真正的去偏操作
    beta_desparse_full = beta_init - H_inv @ unpenalized_grad
    beta_desparse = beta_desparse_full[1:] # Discard the intercept term / 丢弃截距项
    
    # 3. Inject homoscedastic global GDP noise for Stage 2 / 3. 注入阶段二的同方差全局 GDP 噪声
    Delta2 = 2.0 / np.sqrt(n) 
    sigma2 = Delta2 / mu2
    noise = np.random.normal(0, sigma2, size=len(beta_desparse))
    beta_tilde = beta_desparse + noise
    
    # Package as dictionary for server alignment / 打包为字典形式以便中心端对齐
    coeff_dict = {S_minus_k[i]: beta_tilde[i] for i in range(len(S_minus_k))}
    coeffs_perturbed.append(coeff_dict)
    
    print(f"Site {k+1} successfully completed debiasing and injected homoscedastic noise (variance={sigma2:.4f}) / 站点 {k+1} 成功完成去偏并注入同方差噪声 (方差={sigma2:.4f})")

Site 1 successfully completed debiasing and injected homoscedastic noise (variance=0.1633) / 站点 1 成功完成去偏并注入同方差噪声 (方差=0.1633)
Site 2 successfully completed debiasing and injected homoscedastic noise (variance=0.1633) / 站点 2 成功完成去偏并注入同方差噪声 (方差=0.1633)
Site 3 successfully completed debiasing and injected homoscedastic noise (variance=0.1633) / 站点 3 成功完成去偏并注入同方差噪声 (方差=0.1633)


In [18]:
# Step 6: Mirror Statistics & Final FDR / 步骤六：镜像统计量与 FDR 闭环
# What it does (做什么):
# The central server aligns the coefficient vectors, calculates pairwise Mirror Statistics, finds the dynamic threshold τ, computes inclusion rates, and ultimately outputs the finalized set of important features while strictly controlling the FDR at α=0.1.
# 中心服务器对齐稀疏系数向量，计算两两站点间的镜像统计量，寻找动态阈值 τ，计算纳入率，最终输出严格满足 FDR 约束的重要特征集。

In [19]:
p_all = p
beta_mats = []
for k, coeff_dict in enumerate(coeffs_perturbed):
    vec = np.zeros(p_all)
    for j, val in coeff_dict.items():
        vec[j] = val
    beta_mats.append(vec)

pairwise_selections = []
# Pairwise pairing to construct mirror statistics / 两两配对构造镜像统计量
for s in range(K):
    for t in range(s+1, K):
        beta_s = beta_mats[s]
        beta_t = beta_mats[t]
        
        # Core formula: Generalized mirror statistic / 核心公式：广义镜像统计量
        sign_prod = np.sign(beta_s * beta_t)
        M = sign_prod * (np.abs(beta_s) + np.abs(beta_t))
        
        M_abs = np.abs(M)
        thresholds = np.sort(M_abs[M_abs > 1e-8])
        best_tau = np.inf
        
        # Find threshold satisfying conservative FDP estimation / 寻找满足保守 FDP 估计的阈值
        for tau in thresholds:
            left = np.sum(M < -tau)
            right = np.sum(M > tau)
            if right == 0:
                continue
            if left / right <= alpha:
                best_tau = tau
                break
        
        if best_tau != np.inf:
            selected = set(np.where(M > best_tau)[0])
        else:
            selected = set()
        pairwise_selections.append(selected)

# Calculate Inclusion Rate, note the normalization of denominator / 计算纳入率 (Inclusion Rate)，注意分母的规范化
inclusion = np.zeros(p_all)
for sel in pairwise_selections:
    weight = max(len(sel), 1)
    for j in sel:
        inclusion[j] += 1.0 / weight

if len(pairwise_selections) > 0:
    inclusion = inclusion * (2.0 / (K * (K - 1)))

# Final FDR control filtering / 最终 FDR 控制过滤
sorted_idx = np.argsort(inclusion)
prefix_sum = np.cumsum(inclusion[sorted_idx])
m_candidates = np.where(prefix_sum <= alpha)[0]

if len(m_candidates) > 0:
    m_max = m_candidates[-1] + 1
    threshold_inc = inclusion[sorted_idx[m_max-1]]
    final_selected = set(np.where(inclusion > threshold_inc)[0])
else:
    final_selected = set()

# ========== Final Evaluation Metrics ========== / ========== 输出终极评估指标 ==========
print("\n========== Final Algorithm Output ========== / ========== 算法最终输出 ==========")
tp = len(final_selected & set(range(p1)))
fp = len(final_selected) - tp
print(f"True Positives (TP): {tp} / {p1} (Full recovery is perfect) / 真正例 (TP): {tp} / {p1} (全部找回为满分)")
print(f"False Positives (FP): {fp} (Noisy features mistaken as signals) / 假发现 (FP): {fp} (被误认为是信号的噪音特征)")
fdp = fp / len(final_selected) if len(final_selected) > 0 else 0.0
print(f"False Discovery Rate (FDP) = {fdp:.3f} (Target: < {alpha}) / FDP 错误发现率 = {fdp:.3f} (目标: < {alpha})")
print(f"Statistical Power = {tp/p1:.3f} (Target: Close to 1.0) / Power 统计功效 = {tp/p1:.3f} (目标: 接近 1.0)")


========== Final Algorithm Output ========== / ========== 算法最终输出 ==========
True Positives (TP): 5 / 8 (Full recovery is perfect) / 真正例 (TP): 5 / 8 (全部找回为满分)
False Positives (FP): 0 (Noisy features mistaken as signals) / 假发现 (FP): 0 (被误认为是信号的噪音特征)
False Discovery Rate (FDP) = 0.000 (Target: < 0.1) / FDP 错误发现率 = 0.000 (目标: < 0.1)
Statistical Power = 0.625 (Target: Close to 1.0) / Power 统计功效 = 0.625 (目标: 接近 1.0)
